# 01 — Reshape IMF Central Government Debt Panel

**Source:** `data/raw/imf-central_gov_debt_data.xlsx`, sheet `CG_DEBT_GDP`  
**Output:** `data/processed/imf_central_govt_debt_panel.csv`

**What this notebook does:**
1. Load and inspect the raw file — understand its exact layout before touching it.
2. Clean the column structure — identify and isolate year columns, drop pre-1990 data.
3. Reshape wide → long (panel format).
4. Add ISO 3166-1 alpha-3 codes, since the file only contains country names.
5. Data quality summary — coverage, missingness, distribution, suspicious values.
6. Save the cleaned panel.

Every exclusion or coercion is counted and printed. Nothing is silently dropped.

---
### Setup — imports and project root



In [1]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start: Path = Path().resolve()) -> Path:
    """Walk up until config/ is found — works from any working directory."""
    for directory in [start, *start.parents]:
        if (directory / "config").is_dir():
            return directory
    raise FileNotFoundError(
        "Cannot find project root — expected a config/ folder somewhere above this notebook."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import LOG_FORMAT, LOG_DATE_FORMAT

logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    datefmt=LOG_DATE_FORMAT,
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("imf_debt")

INPUT_FILE  = PROJECT_ROOT / "data" / "raw" / "imf-central_gov_debt_data.xlsx"
OUTPUT_DIR  = PROJECT_ROOT / "data" / "processed"
OUTPUT_FILE = OUTPUT_DIR / "imf_central_govt_debt_panel.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

log.info("Project root : %s", PROJECT_ROOT)
log.info("Input file   : %s  (exists: %s)", INPUT_FILE, INPUT_FILE.exists())
log.info("Output file  : %s", OUTPUT_FILE)

2026-04-07 23:11:24 | INFO     | imf_debt | Project root : C:\Users\nduka\OneDrive\Desktop\Claude\sovereign-risk-agent
2026-04-07 23:11:24 | INFO     | imf_debt | Input file   : C:\Users\nduka\OneDrive\Desktop\Claude\sovereign-risk-agent\data\raw\imf-central_gov_debt_data.xlsx  (exists: True)
2026-04-07 23:11:24 | INFO     | imf_debt | Output file  : C:\Users\nduka\OneDrive\Desktop\Claude\sovereign-risk-agent\data\processed\imf_central_govt_debt_panel.csv


---
### Step 1 — Load and inspect the raw file


**Loading strategy:**
- `header=0` — use row 0 as column names.
- `skiprows=[1]` — skip the blank separator row so it doesn't appear in the data.
- `dtype=str` — read everything as strings so no year labels become `1990.0`.
- `na_values=[...]` — tell pandas which strings mean missing. The IMF uses `"no data"` throughout, but we also catch the other common encodings in advance.

In [2]:
NA_STRINGS = ["", "n/a", "N/A", "NA", "--", "...", "no data", "No data"]

raw = pd.read_excel(
    INPUT_FILE,
    sheet_name="CG_DEBT_GDP",
    engine="openpyxl",      # .xlsx format requires openpyxl, NOT xlrd
    dtype=str,
    header=0,
    skiprows=[1],           # skip the blank separator row at index 1
    na_values=NA_STRINGS,
    keep_default_na=False,  # only treat our explicit list as NA
)

print(f"Raw shape: {raw.shape[0]} rows × {raw.shape[1]} columns")
print()
print("First 5 rows, first 10 columns:")
print(raw.iloc[:5, :10].to_string())
print()
print("All column names:")
print(raw.columns.tolist())

Raw shape: 176 rows × 76 columns

First 5 rows, first 10 columns:
  Central Government Debt (Percent of GDP) 1950 1951 1952 1953 1954 1955 1956 1957 1958
0                              Afghanistan  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN
1                                  Albania  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN
2                                  Algeria  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN
3                                   Angola  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN
4                      Antigua and Barbuda  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN

All column names:
['Central Government Debt (Percent of GDP)', 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 

### 1b — Inspect the first column

The first column is labelled `"Central Government Debt (Percent of GDP)"` in the file header.
We need to confirm what it actually contains: country names, ISO codes, or both.
We also need to see the footer rows so we know exactly what to drop.

In [3]:
first_col = raw.columns[0]
print(f"First column label: {repr(first_col)}")
print()

# Show a sample of values in the first column
print("First 10 values in the first column:")
for v in raw[first_col].head(10):
    print(f"  {repr(v)}")

print()
print("Last 5 values in the first column (checking for footer rows):")
for v in raw[first_col].tail(5):
    print(f"  {repr(v)}")

print()
print("Verdict: the first column contains COUNTRY NAMES (not ISO codes).")
print("We will rename it to 'country' and add ISO 3166-1 alpha-3 codes in Step 4.")

First column label: 'Central Government Debt (Percent of GDP)'

First 10 values in the first column:
  'Afghanistan'
  'Albania'
  'Algeria'
  'Angola'
  'Antigua and Barbuda'
  'Argentina'
  'Armenia'
  'Australia'
  'Austria'
  'Azerbaijan'

Last 5 values in the first column (checking for footer rows):
  'Yemen'
  'Zambia'
  'Zimbabwe'
  nan
  '©IMF, 2025'

Verdict: the first column contains COUNTRY NAMES (not ISO codes).
We will rename it to 'country' and add ISO 3166-1 alpha-3 codes in Step 4.


---
### Step 2 — Clean the column structure


1. **Rename** the first column to `"country"`.
2. **Drop non-country rows** — the blank row and the `©IMF, 2025` footer were loaded as data rows. 
3. **Identify year columns** — year headers were read as strings (`"1950"`, …, `"2024"`). We detect them by checking whether the value parses as a 4-digit integer between 1900 and 2100.
4. **Drop pre-1990 year columns** — we only need 1990–2024 to align with the World Bank and WEO datasets.

In [ ]:
df = raw.copy()

# 2.1  Rename first column
df = df.rename(columns={df.columns[0]: "country"})

# 2.2  Drop non-country rows (blank row, ©IMF footer) 
n_before = len(df)
df = df[df["country"].notna()].copy()
df = df[~df["country"].str.startswith("\xa9", na=False)].copy()  # © = \xa9
n_dropped = n_before - len(df)
print(f"Dropped {n_dropped} non-country rows (blank + footer). Remaining: {len(df)}")

# 2.3  Identify year columns
def is_year_col(col: str) -> bool:
    try:
        yr = int(str(col).strip().replace(".0", ""))  # handle "1990.0" if it appears
        return 1900 <= yr <= 2100
    except (ValueError, TypeError):
        return False

all_year_cols  = [c for c in df.columns if is_year_col(c)]
meta_cols      = [c for c in df.columns if not is_year_col(c)]

print(f"\nAll year columns: {len(all_year_cols)}  ({all_year_cols[0]} … {all_year_cols[-1]})")
print(f"Non-year columns: {meta_cols}")

# 2.4  Keep only 1990–2024 
year_cols = [c for c in all_year_cols if 1990 <= int(str(c).strip().replace(".0", "")) <= 2024]
dropped_year_cols = [c for c in all_year_cols if c not in year_cols]

print(f"\nDropping {len(dropped_year_cols)} pre-1990 year columns: {dropped_year_cols[0]} … {dropped_year_cols[-1]}")
print(f"Keeping  {len(year_cols)} year columns: {year_cols[0]} … {year_cols[-1]}")

df = df[["country"] + year_cols].copy()

print(f"\nCleaned shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nCleaned column names:")
print(df.columns.tolist())

Dropped 2 non-country rows (blank + footer). Remaining: 174

All year columns: 75  (1950 … 2024)
Non-year columns: ['country']

Dropping 40 pre-1990 year columns: 1950 … 1989
Keeping  35 year columns: 1990 … 2024

Cleaned shape: 174 rows × 36 columns

Cleaned column names:
['country', 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


---
## Step 3 — Reshape from wide to long (panel format)

**Wide format** (current): 1 row per country, 35 year columns.  
**Panel format** (target): 1 row per country-year, with a single `debt_to_gdp` column.

`pd.melt()` unpivots the year columns: it takes each (country, year) cell and creates  
a separate row. The result has three columns: `country`, `year`, `debt_to_gdp`.

After melting we:
- Convert `year` to integer.
- Convert `debt_to_gdp` to float, using `pd.to_numeric(errors='coerce')` so that any  
  remaining non-numeric strings become `NaN`. We count these explicitly — they are not dropped.
- Sort by country, then year.

In [6]:
# ── 3.1  Melt ─────────────────────────────────────────────────────────────────
panel = df.melt(
    id_vars=["country"],
    value_vars=year_cols,
    var_name="year",
    value_name="debt_to_gdp",
)

# ── 3.2  Convert year to integer ──────────────────────────────────────────────
panel["year"] = panel["year"].astype(str).str.strip().str.replace(".0", "", regex=False)
panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype("Int64")

# ── 3.3  Convert debt_to_gdp to float — count coercions ──────────────────────
# Values already in NA_STRINGS were converted to NaN by read_excel.
# Any residual non-numeric strings (unexpected encoding) are coerced here.
before_na = panel["debt_to_gdp"].isna().sum()

panel["debt_to_gdp"] = pd.to_numeric(
    panel["debt_to_gdp"].astype(str).str.strip().str.replace(",", "", regex=False),
    errors="coerce",
)

after_na      = panel["debt_to_gdp"].isna().sum()
newly_coerced = after_na - before_na

print(f"NaN values BEFORE numeric conversion : {before_na:,}")
print(f"NaN values AFTER  numeric conversion : {after_na:,}")
print(f"Values newly coerced to NaN          : {newly_coerced:,}")
if newly_coerced > 0:
    coerced_sample = panel[panel["debt_to_gdp"].isna() & panel["debt_to_gdp"].notna()]
    print("  (these were non-numeric strings not caught by na_values — review if > 0)")

# ── 3.4  Sort ─────────────────────────────────────────────────────────────────
panel = panel.sort_values(["country", "year"]).reset_index(drop=True)

print(f"\nPanel shape: {panel.shape[0]:,} rows × {panel.shape[1]} columns")
print("\nFirst 10 rows:")
print(panel.head(10).to_string(index=False))

NaN values BEFORE numeric conversion : 500
NaN values AFTER  numeric conversion : 500
Values newly coerced to NaN          : 0

Panel shape: 6,090 rows × 3 columns

First 10 rows:
    country  year  debt_to_gdp
Afghanistan  1990          NaN
Afghanistan  1991          NaN
Afghanistan  1992          NaN
Afghanistan  1993          NaN
Afghanistan  1994          NaN
Afghanistan  1995          NaN
Afghanistan  1996          NaN
Afghanistan  1997          NaN
Afghanistan  1998          NaN
Afghanistan  1999          NaN


---
## Step 4 — Add ISO 3166-1 alpha-3 country codes

The file contains **country names only** (confirmed in Step 1). Our other datasets  
(World Bank, IMF WEO) use ISO 3-letter codes as the join key, so we need to add them.

**Strategy:**
1. Try an exact lookup via `pycountry.countries.get(name=...)` first.
2. Fall back to `pycountry.countries.search_fuzzy(...)` for slight name variations.
3. Apply a **manual override dictionary** for IMF-specific names that neither method  
   handles correctly (e.g. `"Bahamas, The"`, `"Lao P.D.R."`, `"Türkiye, Republic of"`,  
   territories like `"West Bank and Gaza"`, etc.).
4. Print any country names that still cannot be matched after all three methods,  
   so you can review and add overrides if needed.

Countries that fail all three methods receive `iso3 = NaN` — they are **not dropped**.

In [10]:
# Install pycountry if not already present
%pip install pycountry --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pycountry

# Manual overrides: IMF country names → ISO 3166-1 alpha-3 
# These are names that pycountry cannot match exactly or fuzzily because the IMF
# uses its own naming conventions, includes territories, or has encoding quirks.
MANUAL_ISO = {
    # IMF comma-inverted names
    "Bahamas, The":                          "BHS",
    "Congo, Republic of":                    "COG",
    "Congo, Republic of ":                   "COG",   # trailing space variant
    "Congo, Democratic Republic of the":     "COD",
    "Egypt, Arab Republic of":               "EGY",
    "Gambia, The":                           "GMB",
    "Iran, Islamic Republic of":             "IRN",
    "Korea, Republic of":                    "KOR",
    "Micronesia, Federated States of":       "FSM",
    "South Sudan, Republic of":              "SSD",
    "Syria":                                 "SYR",
    "Tanzania, United Republic of":          "TZA",
    "Venezuela, Republica Bolivariana de":   "VEN",
    "Venezuela, Rep\u00fablica Bolivariana de": "VEN",
    "Yemen, Republic of":                    "YEM",
    # IMF abbreviations
    "Kyrgyz Republic":                       "KGZ",
    "Lao P.D.R.":                            "LAO",
    "Slovak Republic":                       "SVK",
    # Special Administrative Regions
    "Hong Kong SAR":                         "HKG",
    "Macao SAR":                             "MAC",
    # Territories / observer states
    "Kosovo":                                "XKX",   # not in ISO 3166-1; UNSD code
    "Taiwan Province of China":              "TWN",
    "West Bank and Gaza":                    "PSE",
    # Encoding / name variants
    "North Macedonia":                       "MKD",
    "S\u00e3o Tom\u00e9 and Pr\u00edncipe": "STP",
    "T\u00fcrkiye":                          "TUR",
    "T\u00fcrkiye, Republic of":             "TUR",
    "T\ufffdrkiye, Republic of":             "TUR",   # mojibake variant in this file
    "China, People s Republic of":           "CHN",
    "China, People's Republic of":           "CHN",
    "China, People\u2019s Republic of":      "CHN",
}


def lookup_iso3(name: str) -> str | None:
    """Return ISO 3166-1 alpha-3 code for a country name, or None if not found."""
    cleaned = str(name).strip()

    # 1. Manual override (fastest, most reliable for known problem names)
    if cleaned in MANUAL_ISO:
        return MANUAL_ISO[cleaned]

    # 2. Exact pycountry lookup
    result = pycountry.countries.get(name=cleaned)
    if result:
        return result.alpha_3

    # 3. Fuzzy pycountry lookup
    try:
        matches = pycountry.countries.search_fuzzy(cleaned)
        if matches:
            return matches[0].alpha_3
    except LookupError:
        pass

    return None


# Build the mapping on unique country names (faster than per-row lookup)
unique_countries = panel["country"].unique()
iso_map          = {name: lookup_iso3(name) for name in unique_countries}

# Apply to panel
panel["iso3"] = panel["country"].map(iso_map)

# Report results
matched   = {k: v for k, v in iso_map.items() if v is not None}
unmatched = {k: v for k, v in iso_map.items() if v is None}

print(f"Countries matched   : {len(matched)} / {len(iso_map)}")
print(f"Countries unmatched : {len(unmatched)}")

if unmatched:
    print("\n--- UNMATCHED COUNTRY NAMES (iso3 set to NaN) ---")
    for name in sorted(unmatched):
        print(f"  {repr(name)}")
    print("\nAction required: add these to MANUAL_ISO above, or accept NaN.")
else:
    print("\nAll countries successfully matched to an ISO3 code.")

Countries matched   : 174 / 174
Countries unmatched : 0

All countries successfully matched to an ISO3 code.


---
## Step 5 — Data quality summary

Before saving, we run four checks:

1. **Coverage** — unique countries and year range.
2. **Missingness** — overall and broken down by decade.
3. **Distribution** — mean, median, min, max, std of `debt_to_gdp`.
4. **Suspicious values** — debt below 0% or above 300% GDP. These can be genuine  
   (e.g. Japan, Zimbabwe during hyperinflation) — we flag them but do not remove them.

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

# 5.1  Coverage
print("=" * 60)
print("5.1  COVERAGE")
print("=" * 60)
print(f"  Unique countries : {panel['country'].nunique()}")
print(f"  ISO3 codes       : {panel['iso3'].nunique()} (excludes NaN)")
print(f"  Year range       : {panel['year'].min()} – {panel['year'].max()}")
print(f"  Total rows       : {len(panel):,}")

# 5.2  Missingness
print()
print("=" * 60)
print("5.2  MISSINGNESS — debt_to_gdp")
print("=" * 60)
total_missing  = panel["debt_to_gdp"].isna().sum()
total_pct      = total_missing / len(panel) * 100
print(f"  Overall: {total_missing:,} / {len(panel):,} rows missing  ({total_pct:.1f}%)")

print()
print("  By decade:")
decades = {
    "1990s (1990–1999)": (1990, 1999),
    "2000s (2000–2009)": (2000, 2009),
    "2010s (2010–2019)": (2010, 2019),
    "2020s (2020–2024)": (2020, 2024),
}
for label, (yr_start, yr_end) in decades.items():
    decade_mask = panel["year"].between(yr_start, yr_end)
    decade_df   = panel[decade_mask]
    d_missing   = decade_df["debt_to_gdp"].isna().sum()
    d_total     = len(decade_df)
    d_pct       = d_missing / d_total * 100 if d_total > 0 else float("nan")
    print(f"    {label}: {d_missing:,} / {d_total:,} missing  ({d_pct:.1f}%)")

# 5.3  Distribution
print("=" * 60)
print("5.3  DISTRIBUTION — debt_to_gdp (% GDP)")
print("=" * 60)
desc = panel["debt_to_gdp"].describe()
print(f"  Count  : {desc['count']:,.0f} non-null observations")
print(f"  Mean   : {desc['mean']:.2f}%")
print(f"  Median : {desc['50%']:.2f}%")
print(f"  Std    : {desc['std']:.2f}%")
print(f"  Min    : {desc['min']:.2f}%")
print(f"  Max    : {desc['max']:.2f}%")

# 5.4  Suspicious values 
print()
print("=" * 60)
print("5.4  SUSPICIOUS VALUES (debt < 0% or > 300%)")
print("=" * 60)
suspicious = panel[panel["debt_to_gdp"].notna() & (
    (panel["debt_to_gdp"] < 0) | (panel["debt_to_gdp"] > 300)
)]

if suspicious.empty:
    print("  None found — all non-null values are in the range [0, 300].")
else:
    print(f"  Found {len(suspicious)} suspicious rows:")
    print()
    print(suspicious[["country", "year", "debt_to_gdp"]]
          .sort_values("debt_to_gdp", ascending=False)
          .to_string(index=False))
    print()
    print("  Note: these rows are retained — verify against primary sources before modelling.")

5.1  COVERAGE
  Unique countries : 174
  ISO3 codes       : 174 (excludes NaN)
  Year range       : 1990 – 2024
  Total rows       : 6,090

5.2  MISSINGNESS — debt_to_gdp
  Overall: 500 / 6,090 rows missing  (8.2%)

  By decade:
    1990s (1990–1999): 339 / 1,740 missing  (19.5%)
    2000s (2000–2009): 83 / 1,740 missing  (4.8%)
    2010s (2010–2019): 26 / 1,740 missing  (1.5%)
    2020s (2020–2024): 52 / 870 missing  (6.0%)

5.3  DISTRIBUTION — debt_to_gdp (% GDP)
  Count  : 5,590 non-null observations
  Mean   : 55.47%
  Median : 45.15%
  Std    : 47.40%
  Min    : 0.00%
  Max    : 677.18%

5.4  SUSPICIOUS VALUES (debt < 0% or > 300%)
  Found 25 suspicious rows:

              country  year  debt_to_gdp
São Tomé and Príncipe  1996       677.18
              Liberia  2003       658.22
              Liberia  2004       543.14
              Liberia  2005       534.89
São Tomé and Príncipe  1999       526.18
São Tomé and Príncipe  1997       495.95
                Sudan  1992       495.2

---
## Step 6 — Save the cleaned panel

Column order in the output CSV:

| Column | Type | Description |
|--------|------|-------------|
| `country` | string | Country name as it appears in the IMF source file |
| `iso3` | string | ISO 3166-1 alpha-3 code (NaN if unmatched) |
| `year` | integer | Calendar year (1990–2024) |
| `debt_to_gdp` | float | Central government debt as % of GDP |

The pandas index is excluded (`index=False`).

In [ ]:
# Reorder columns and save
output_cols = ["country", "iso3", "year", "debt_to_gdp"]
panel_out   = panel[output_cols].copy()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
panel_out.to_csv(OUTPUT_FILE, index=False)

size_kb = OUTPUT_FILE.stat().st_size / 1024

print("=" * 60)
print("SAVED")
print("=" * 60)
print(f"  File      : {OUTPUT_FILE}")
print(f"  Shape     : {panel_out.shape[0]:,} rows × {panel_out.shape[1]} columns")
print(f"  Columns   : {panel_out.columns.tolist()}")
print(f"  File size : {size_kb:.1f} KB")
print()
print("Preview (first 10 rows of saved file):")
print(pd.read_csv(OUTPUT_FILE).head(10).to_string(index=False))

SAVED
  File      : C:\Users\nduka\OneDrive\Desktop\Claude\sovereign-risk-agent\data\processed\imf_central_govt_debt_panel.csv
  Shape     : 6,090 rows × 4 columns
  Columns   : ['country', 'iso3', 'year', 'debt_to_gdp']
  File size : 174.1 KB

Preview (first 10 rows of saved file):
    country iso3  year  debt_to_gdp
Afghanistan  AFG  1990          NaN
Afghanistan  AFG  1991          NaN
Afghanistan  AFG  1992          NaN
Afghanistan  AFG  1993          NaN
Afghanistan  AFG  1994          NaN
Afghanistan  AFG  1995          NaN
Afghanistan  AFG  1996          NaN
Afghanistan  AFG  1997          NaN
Afghanistan  AFG  1998          NaN
Afghanistan  AFG  1999          NaN


: 